# 00 · Funções e parâmetros compartilhados (versão 7)

Carregado pelos demais notebooks com `%run ./00_funcoes.ipynb`.

**Versão 7:** mesmos dados e cenários da v3 (maio a setembro de 2026; teste em agosto e 1–20/09; histórico de 3 meses × 1 mês). Muda o método de decisão: o momento do registro é **Normal** se se enquadra em algum dos **regimes normais admissíveis** para aquele contexto (tipo de dia, hora); os regimes vêm de k-means nos indicadores do histórico, cada um com mediana/MAD próprias. O limite de cada indicador passa a ser o **percentil 95** (v3: 99), com c = 1 fixo.

**Pergunta:** *no instante em que o incidente é registrado, a API (recurso) está operando dentro de um de seus regimes normais? Se não, qual indicador saiu do padrão (erros de negócio 4xx, erros de sistema 5xx ou latência) e desde quando?* A abrangência (consumidor específico ou generalizado) é informação complementar.

Técnicas usadas (6):
1. **Kruskal-Wallis** com Bonferroni: o comportamento muda com a hora?
2. **k-means + silhouette**: regimes normais de cada recurso, aprendidos só no histórico de cada mês de teste;
3. **Mediana e MAD por regime** + **admissibilidade por contexto** (regime presente em ≥ 5% das janelas do tipo de dia × hora, ± 1 h);
4. **Limite por atributo** (percentil 95 fora da amostra, validação cruzada): decisão Normal/Residual e indicador responsável;
5. **Expansão retroativa**: estimativa do início;
6. **IC de Wilson + McNemar**: confiabilidade dos resultados e comparação com baselines.


In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

## 1. Parâmetros

In [ ]:
VERSAO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJETO = VERSAO.parent
ARQUIVO_DADOS = PROJETO / "DADOS" / "novoCj" / "APIs-CjAnterior-20260501-20260920-mes_hash.csv"
RESULTADOS = VERSAO / "resultados"
FIGURAS = RESULTADOS / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

FUSO = "America/Sao_Paulo"               # TS_EVT vem em UTC
FERIADOS = ["2026-05-01", "2026-06-04", "2026-09-07"]  # Trabalho, Corpus Christi, Independência = dia não útil
JANELA_PERFIL = "1h"                     # o perfil normal é modelado em janelas de 1 h
MESES_TESTE = {"ago": (pd.Timestamp("2026-08-01"), pd.Timestamp("2026-09-01")),
               "set": (pd.Timestamp("2026-09-01"), pd.Timestamp("2026-09-21"))}
HISTORICOS = {"3m": 3, "1m": 1}          # meses de histórico antes de cada mês de teste
HIST_PRINCIPAL = "3m"
MIN_HIST_DIAS = 28                       # dias de dados exigidos no histórico de 1 mês para o recurso entrar no teste
TREINO = (pd.Timestamp("2026-05-01"), pd.Timestamp("2026-09-01"))  # caracterização dos regimes (notebook 03)


def periodo_treino(mes, hist):
    ini = MESES_TESTE[mes][0]
    return (ini - pd.DateOffset(months=HISTORICOS[hist]), ini)

FAIXAS_L = [15, 30, 60]                  # minutos antes do registro analisados
L_PRINCIPAL = 15
BANDAS = [(0, 15), (15, 30), (30, 60), (60, 120), (120, 180), (180, 240)]  # expansão retroativa (min antes)
H_MAX = 240

ATRIBUTOS = ["log_vol", "tx_4xx", "tx_5xx", "log_lat_p95"]
INDICADOR = {"log_vol": "volume", "tx_4xx": "erros 4xx", "tx_5xx": "erros 5xx", "log_lat_p95": "latência p95"}
DIRECAO = {"log_vol": "ambos", "tx_4xx": "alta", "tx_5xx": "alta", "log_lat_p95": "alta"}
ESCALA_MIN = pd.Series({"log_vol": 0.10, "tx_4xx": 0.01, "tx_5xx": 0.01, "log_lat_p95": 0.05})

META_ALARMES = None                      # v7: c = 1 fixo (com meta de alarmes, c subiria e anularia o percentil 95)
METAS_CURVA = [0.01, 0.02, 0.05, 0.10]   # pontos de operação relatados (c ajustado para a meta)
Q_LIMITE = 0.95                          # v7: percentil 95 (v3: 99) das variações de cada atributo
LIMITE_MIN = 3.0                         # limite mínimo (escala z)

K_REGIMES = range(2, 9)                  # número de regimes testado no k-means (silhouette)
REGIME_MIN = 0.01                        # regime com ao menos 1% das janelas do histórico
ADMISSAO_MIN = 0.05                      # regime admissível no contexto se ocorre em ≥ 5% das suas janelas
TOLERANCIA_H = 1                         # ... ou nas horas vizinhas (± 1 h), para lotes que atrasam

MOTIVOS = {  # motivo -> (indicador esperado, abrangência, fatores baixa/média/alta)
    "4xx_geral": ("erros 4xx", "generalizado", {"baixa": 0.05, "media": 0.15, "alta": 0.40}),
    "4xx_cliente": ("erros 4xx", "consumidor específico", {"baixa": 0.30, "media": 0.60, "alta": 0.90}),
    "5xx_geral": ("erros 5xx", "generalizado", {"baixa": 0.05, "media": 0.15, "alta": 0.40}),
    "5xx_cliente": ("erros 5xx", "consumidor específico", {"baixa": 0.30, "media": 0.60, "alta": 0.90}),
    "latencia": ("latência p95", "generalizado", {"baixa": 1.5, "media": 2.5, "alta": 4.0, "muito_alta": 10.0}),
}
NOMES_MOTIVO = {"4xx_geral": "4xx generalizado", "4xx_cliente": "4xx de 1 consumidor",
                "5xx_geral": "5xx generalizado", "5xx_cliente": "5xx de 1 consumidor", "latencia": "latência"}
INTENSIDADES = ["baixa", "media", "alta", "muito_alta"]  # "muito alta" (×10) só para latência
ATRASOS = [15, 30, 60, 120]              # minutos entre o início da degradação e o registro
REPETICOES = 5                           # cenários por recurso × motivo × intensidade × atraso
SEMENTE = 0

## 2. Preparação dos dados
Cada linha da base é um **agregado por minuto × recurso × código HTTP × cliente**, com o volume de transações e o tempo médio de execução naquele minuto. Não há registros por requisição.

In [ ]:
COLUNAS = {"TS_EVT": "ts", "CD_RCS_CAT": "recurso", "CD_RPST": "status",
           "QT_TTL_TRAN": "qtd", "QT_TMP_MEDI_EXEA": "tmp", "CD_CLI": "cliente"}
NOMES_DIA = {"util": "dia útil", "nao_util": "não útil"}


def carregar(caminho=ARQUIVO_DADOS, fuso=FUSO):
    """Lê o CSV e converte TS_EVT de UTC para o horário de Brasília."""
    df = pd.read_csv(caminho, usecols=list(COLUNAS), parse_dates=["TS_EVT"]).rename(columns=COLUNAS)
    df["ts"] = df["ts"].dt.tz_localize("UTC").dt.tz_convert(fuso).dt.tz_localize(None)
    return df


def consolidar(df):
    """Linhas com a mesma chave (minuto, recurso, status, cliente): soma o volume, latência média ponderada."""
    df = df.assign(tmp_total=df["qtd"] * df["tmp"])
    out = df.groupby(["ts", "recurso", "status", "cliente"], as_index=False, sort=False).agg(
        qtd=("qtd", "sum"), tmp_total=("tmp_total", "sum"))
    out["tmp"] = out["tmp_total"] / out["qtd"]
    return out.drop(columns="tmp_total")


def arquivo_codigos():
    return RESULTADOS / "01_codigos.json"


def codigos():
    """Mapa recurso -> E01..E12 (ordem decrescente de volume), gerado no notebook 01."""
    arq = arquivo_codigos()
    return {int(k): v for k, v in json.loads(arq.read_text()).items()} if arq.exists() else {}


def rot(r):
    return codigos().get(int(r), str(r))


def tipo_dia(ts, feriados=FERIADOS):
    feriado = ts.dt.normalize().isin(pd.to_datetime(list(feriados)))
    return np.where((ts.dt.dayofweek >= 5) | feriado, "nao_util", "util")


def p95_ponderado(df, chaves, q=0.95):
    """Percentil ponderado pelo volume das latências médias por minuto × cliente (só 2xx)."""
    d = df.loc[df["status"] // 100 == 2, chaves + ["tmp", "qtd"]].sort_values(chaves + ["tmp"])
    g = d.groupby(chaves, sort=False)
    cum = g["qtd"].cumsum()
    tot = g["qtd"].transform("sum")
    alvo = d[cum >= q * tot]
    return alvo.groupby(chaves, sort=False)["tmp"].first()


def agregar(df, janela, feriados=FERIADOS, limites=None):
    """Agrega por recurso × janela. O volume fica na escala horária (× 60/minutos da janela)."""
    minutos = pd.Timedelta(janela) / pd.Timedelta("1min")
    df = df.assign(janela=df["ts"].dt.floor(janela), fam=df["status"] // 100)
    for f in (2, 4, 5):
        df[f"q{f}"] = df["qtd"].where(df["fam"] == f, 0)
    niveis = ["recurso", "janela"]
    jan = df.groupby(niveis)[["qtd", "q2", "q4", "q5"]].sum().rename(columns={"qtd": "vol"})
    cli = df.groupby(["recurso", "janela", "cliente"], sort=False)[["qtd", "q4", "q5"]].sum()
    jan = jan.join(cli.groupby(level=niveis).agg(n_cli=("qtd", "size"), max_cli=("qtd", "max")))
    jan["lat_p95"] = p95_ponderado(df, niveis)
    if limites is None:
        g = df.groupby("recurso")["ts"].agg(["min", "max"])
        limites = pd.DataFrame({"inicio": g["min"].dt.floor(janela), "fim": (g["max"] + pd.Timedelta("1min")).dt.floor(janela)})
    grades = [pd.MultiIndex.from_product([[r], pd.date_range(l.inicio, l.fim - pd.Timedelta(janela), freq=janela)], names=niveis)
              for r, l in limites.iterrows()]
    jan = jan.reindex(grades[0].append(grades[1:]))
    cont = ["vol", "q2", "q4", "q5", "n_cli", "max_cli"]
    jan[cont] = jan[cont].fillna(0)
    jan = jan.reset_index()
    vol = jan["vol"].replace(0, np.nan)
    jan["tx_4xx"] = (jan["q4"] / vol).fillna(0)
    jan["tx_5xx"] = (jan["q5"] / vol).fillna(0)
    jan["top1_vol"] = (jan["max_cli"] / vol).fillna(0)
    jan["log_vol"] = np.log1p(jan["vol"] * 60 / minutos)
    jan["log_lat_p95"] = np.log1p(jan["lat_p95"])
    meio = jan["janela"] + pd.Timedelta(minutes=minutos / 2)
    jan["hora"] = meio.dt.hour
    jan["tipo_dia"] = tipo_dia(meio, feriados)
    jan["minutos"] = int(minutos)
    return jan.sort_values(niveis).reset_index(drop=True)


def recortar(jan, periodo, col="janela"):
    return jan[(jan[col] >= periodo[0]) & (jan[col] < periodo[1])].copy()

## 3. Períodos arbitrários a partir dos dados por minuto
Para a consulta no instante do registro, o período [t − L, t) é agregado diretamente dos registros por minuto (janela deslizante, sem depender da hora cheia).

In [ ]:
T0 = pd.Timestamp("2026-01-01")


def minutos_desde(ts):
    return ((pd.to_datetime(ts) - T0) / pd.Timedelta("1min")).astype(np.int64)


class Brutos:
    """Registros por minuto de cada recurso em arrays ordenados, para fatiar períodos rapidamente."""

    def __init__(self, df):
        cli_cod = {c: i for i, c in enumerate(df["cliente"].unique())}
        self.n_cli_total = len(cli_cod)
        self.por_recurso = {}
        for r, g in df.sort_values("ts").groupby("recurso"):
            self.por_recurso[r] = dict(
                t=minutos_desde(g["ts"]).to_numpy(), status=g["status"].to_numpy(),
                qtd=g["qtd"].to_numpy().astype(np.int64), tmp=g["tmp"].to_numpy().astype(float),
                cli=g["cliente"].map(cli_cod).to_numpy())
        self.inicio = {r: pd.Timestamp(T0 + pd.Timedelta(minutes=int(a["t"].min()))) for r, a in self.por_recurso.items()}
        self.fim = {r: pd.Timestamp(T0 + pd.Timedelta(minutes=int(a["t"].max()) + 1)) for r, a in self.por_recurso.items()}

    def fatia(self, r, ini, fim):
        a = self.por_recurso[r]
        i, j = np.searchsorted(a["t"], [minutos_desde(pd.Series([ini]))[0], minutos_desde(pd.Series([fim]))[0]])
        return {k: v[i:j].copy() for k, v in a.items()}


def agregar_periodo(f, ini, fim):
    """Atributos de um período [ini, fim) a partir de uma fatia de registros (``f``)."""
    t_ini, t_fim = minutos_desde(pd.Series([ini]))[0], minutos_desde(pd.Series([fim]))[0]
    m = (f["t"] >= t_ini) & (f["t"] < t_fim) & (f["qtd"] > 0)
    qtd, fam, tmp, cli = f["qtd"][m], f["status"][m] // 100, f["tmp"][m], f["cli"][m]
    minutos = t_fim - t_ini
    vol = qtd.sum()
    out = dict(vol=vol, minutos=minutos)
    q4, q5 = qtd[fam == 4].sum(), qtd[fam == 5].sum()
    out["tx_4xx"] = q4 / vol if vol else 0.0
    out["tx_5xx"] = q5 / vol if vol else 0.0
    ok = fam == 2
    if ok.any():
        o = np.argsort(tmp[ok])
        w, v = qtd[ok][o], tmp[ok][o]
        out["lat_p95"] = v[np.searchsorted(np.cumsum(w), 0.95 * w.sum())]
    else:
        out["lat_p95"] = np.nan
    out["log_vol"] = np.log1p(vol * 60 / minutos)
    out["log_lat_p95"] = np.log1p(out["lat_p95"])
    # abrangência: cliente com mais erros de cada família e sua participação no volume
    vol_cli = np.bincount(cli, weights=qtd) if len(cli) else np.zeros(1)
    out["n_cli"] = int((vol_cli > 0).sum())
    for fam_x, q in ((4, q4), (5, q5)):
        err = np.bincount(cli[fam == fam_x], weights=qtd[fam == fam_x], minlength=len(vol_cli)) if q else np.zeros(1)
        top = int(np.argmax(err)) if q else 0
        out[f"n_cli_{fam_x}xx"] = int((err > 0).sum())
        out[f"top_err_{fam_x}xx"] = err[top] / q if q else 0.0
        out[f"top_vol_{fam_x}xx"] = vol_cli[top] / vol if q and vol else 0.0
    ts_meio = pd.Timestamp(ini) + (pd.Timestamp(fim) - pd.Timestamp(ini)) / 2
    out["hora"] = ts_meio.hour
    out["tipo_dia"] = tipo_dia(pd.Series([ts_meio]))[0]
    return out

## 4. Regimes normais (proposto) e comparações
- **Regimes normais (proposto):** para cada recurso, k-means nos 4 indicadores das janelas de 1 h do histórico (limitados aos percentis 1 e 99; k de 2 a 8 por silhouette; regimes com ao menos 1% das janelas). Cada regime tem mediana, MAD e faixa p5–p95 próprias. Um regime é **admissível** num contexto (tipo de dia × hora) se ocorreu em ≥ 5% das janelas desse contexto ou das horas vizinhas (± 1 h). O período é comparado ao regime admissível em que melhor se encaixa.
- **Perfil por contexto:** mediana e MAD por recurso × tipo de dia × hora (método proposto da v3/v5: um único normal por contexto).
- **Regimes sem contexto:** os mesmos regimes, mas qualquer um vale a qualquer hora.
- **z-score global (baseline):** mediana e MAD sem contexto (um perfil por recurso).
- **Semana anterior (baseline):** diferença para o mesmo período 7 dias antes, dividida pela variação típica dessas diferenças.

Faixas de 15 e 30 min são comparadas com os modelos de 1 h: o volume é convertido para a escala horária; taxas e latência p95 são usadas diretamente.


In [ ]:
def z_direcional(z):
    return pd.DataFrame({a: (z[a].abs() if DIRECAO[a] == "ambos" else z[a].clip(lower=0)) for a in ATRIBUTOS},
                        index=z.index).fillna(0)


def _mad(x, chaves, mediana):
    x = x.set_index(chaves)[ATRIBUTOS]
    mad = (x - mediana.reindex(x.index).to_numpy()).abs().groupby(level=chaves).median()
    return (1.4826 * mad).clip(lower=ESCALA_MIN, axis=1)


class Perfil:
    """Mediana/MAD por recurso × contexto. ``contexto=()`` = z-score global."""

    def __init__(self, contexto=("tipo_dia", "hora")):
        self.chaves = ["recurso", *contexto]

    def ajustar(self, jan1h, periodos=None):
        g = jan1h.groupby(self.chaves)[ATRIBUTOS]
        self.mediana = g.median()
        self.escala = _mad(jan1h, self.chaves, self.mediana)
        self.q05, self.q95 = g.quantile(0.05), g.quantile(0.95)
        return self

    def referencia(self, p, limite=None):
        idx = pd.MultiIndex.from_frame(p[self.chaves]) if len(self.chaves) > 1 else pd.Index(p["recurso"])
        return [t.reindex(idx).set_axis(p.index) for t in (self.mediana, self.escala, self.q05, self.q95)]

    def z(self, p, limite=None):
        med, esc, _, _ = self.referencia(p)
        return (p[ATRIBUTOS] - med) / esc


class SemanaAnterior:
    """z da diferença para o mesmo período 7 dias antes (colunas ``ant_<atributo>``)."""

    def ajustar(self, jan1h, periodos):
        d = periodos[ATRIBUTOS].to_numpy() - periodos[[f"ant_{a}" for a in ATRIBUTOS]].to_numpy()
        d = pd.DataFrame(d, columns=ATRIBUTOS, index=periodos.index).assign(recurso=periodos["recurso"].to_numpy()).dropna()
        mad = d.groupby("recurso")[ATRIBUTOS].agg(lambda s: (s - s.median()).abs().median())
        self.escala = (1.4826 * mad).clip(lower=ESCALA_MIN, axis=1)
        return self

    def z(self, p, limite=None):
        d = p[ATRIBUTOS].to_numpy() - p[[f"ant_{a}" for a in ATRIBUTOS]].to_numpy()
        return pd.DataFrame(d, columns=ATRIBUTOS, index=p.index) / self.escala.reindex(p["recurso"]).to_numpy()


def faixas_horas(horas):
    horas = sorted(horas)
    if not horas:
        return ""
    partes, ini, ant = [], horas[0], horas[0]
    for h in horas[1:] + [None]:
        if h is not None and h == ant + 1:
            ant = h
            continue
        partes.append(f"{ini}h" if ini == ant else f"{ini}–{ant}h")
        if h is not None:
            ini = ant = h
    return ", ".join(partes)


def nomear(g, rotulos):
    """Nome de cada regime: nível de volume, indicadores típicos, tipo de dia e horas em que predomina."""
    g = g.assign(regime=rotulos)
    ordem_vol = g.groupby("regime")["vol"].median().rank(method="first")
    k = len(ordem_vol)
    nomes = {}
    for c, gg in g.groupby("regime"):
        nivel = "alto" if ordem_vol[c] > 2 * k / 3 else ("baixo" if ordem_vol[c] <= k / 3 else "médio")
        extras = []
        if gg["tx_4xx"].median() > max(0.05, 2 * g["tx_4xx"].median()):
            extras.append("4xx alto")
        if gg["tx_5xx"].median() > max(0.02, 3 * g["tx_5xx"].median()):
            extras.append("5xx alto")
        if gg["log_lat_p95"].median() > g["log_lat_p95"].median() + 0.4:
            extras.append("latência alta")
        dias = gg["tipo_dia"].value_counts(normalize=True)
        dia = NOMES_DIA[dias.index[0]] if dias.iloc[0] >= 0.7 else "todos os dias"
        frac_hora = g.groupby("hora")["regime"].apply(lambda s: (s == c).mean())
        horas = faixas_horas(list(frac_hora[frac_hora >= 0.5].index)) or "esporádico"
        nomes[int(c)] = f"R{c + 1}: volume {nivel}{', ' + ', '.join(extras) if extras else ''} · {dia} · {horas}"
    return nomes


_CACHE_KMEANS = {}


def kmeans_regimes(g):
    """k-means nos indicadores (limitados aos percentis 1 e 99), k ∈ K_REGIMES por silhouette, regimes com
    ao menos REGIME_MIN das janelas; rótulos ordenados pelo volume mediano (R1 = menor volume)."""
    chave = (int(g["recurso"].iloc[0]), int(pd.util.hash_pandas_object(g["janela"], index=False).sum()))
    if chave in _CACHE_KMEANS:
        return _CACHE_KMEANS[chave]
    x = g[ATRIBUTOS].copy()
    x["log_lat_p95"] = x["log_lat_p95"].fillna(x["log_lat_p95"].median())
    x = StandardScaler().fit_transform(x.clip(x.quantile(0.01), x.quantile(0.99), axis=1).fillna(0))
    n_min = max(int(np.ceil(REGIME_MIN * len(x))), 5)
    melhor = (np.nan, np.zeros(len(x), int))
    for k in K_REGIMES:
        if k * n_min > len(x):
            break
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rot_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(x)
        if np.bincount(rot_k, minlength=k).min() < n_min or len(np.unique(rot_k)) < 2:
            continue
        s = silhouette_score(x, rot_k, sample_size=min(len(x), 4000), random_state=0)
        if np.isnan(melhor[0]) or s > melhor[0]:
            melhor = (s, rot_k)
    s, rot_k = melhor
    ordem = pd.Series(g["log_vol"].to_numpy()).groupby(rot_k).median().sort_values().index
    rot_k = pd.Series(rot_k).map({int(c): i for i, c in enumerate(ordem)}).to_numpy()
    _CACHE_KMEANS[chave] = (rot_k, s)
    return rot_k, s


class Regimes:
    """Regimes normais de cada recurso (k-means nos indicadores das janelas de 1 h do histórico), cada um com
    mediana, MAD e faixa p5–p95 próprias. ``contextual=True``: só valem os regimes **admissíveis** no
    (tipo de dia, hora) do período, isto é, que ocorreram em ≥ ADMISSAO_MIN das janelas desse contexto
    (ou das horas vizinhas, ± TOLERANCIA_H). O período é comparado ao regime admissível em que melhor se encaixa
    (menor razão desvio/limite)."""

    escolhe_regime = True
    DIAS = ("util", "nao_util")

    def __init__(self, contextual=True):
        self.contextual = contextual

    def ajustar(self, jan1h, periodos=None):
        self.rec = {}
        for r, g in jan1h.groupby("recurso"):
            g = g.reset_index(drop=True)
            if len(g) < 24:
                continue
            rot_k, s = kmeans_regimes(g)
            K = int(rot_k.max()) + 1
            x = g[ATRIBUTOS]
            gr = x.groupby(rot_k)
            med = gr.median().reindex(range(K))
            mad = (x - med.loc[rot_k].to_numpy()).abs().groupby(rot_k).median().reindex(range(K))
            esc = (1.4826 * mad).clip(lower=ESCALA_MIN, axis=1)
            frac = pd.crosstab([g["tipo_dia"], g["hora"]], rot_k, normalize="index").reindex(columns=range(K), fill_value=0)
            grade = pd.MultiIndex.from_product([self.DIAS, range(24)], names=["tipo_dia", "hora"])
            frac = frac.reindex(grade)
            adm = (frac.fillna(0).to_numpy() >= ADMISSAO_MIN).reshape(2, 24, K)
            if TOLERANCIA_H:
                adm = adm | np.roll(adm, TOLERANCIA_H, axis=1) | np.roll(adm, -TOLERANCIA_H, axis=1)
            vazio = ~adm.any(axis=2)
            adm[vazio] = True  # contexto sem regime admissível (sem dados): todos os regimes valem
            self.rec[r] = dict(med=med.to_numpy(), esc=esc.to_numpy(), q05=gr.quantile(0.05).reindex(range(K)).to_numpy(),
                               q95=gr.quantile(0.95).reindex(range(K)).to_numpy(), adm=adm, frac=frac, k=K, silhouette=s,
                               nomes=nomear(g, rot_k), n=len(g))
        return self

    def _admissiveis(self, m, g):
        if not self.contextual:
            return np.ones((len(g), m["k"]), bool)
        td = np.where(g["tipo_dia"].to_numpy() == "util", 0, 1)
        return m["adm"][td, g["hora"].to_numpy().astype(int)]

    def _escolha(self, p, limite=None):
        """Para cada período: regime escolhido (−1 se o recurso não tem modelo) e z em relação a ele."""
        z = pd.DataFrame(np.nan, index=p.index, columns=ATRIBUTOS)
        reg = pd.Series(-1, index=p.index)
        sentido = np.array([DIRECAO[a] == "ambos" for a in ATRIBUTOS])
        for r, g in p.groupby("recurso"):
            m = self.rec.get(r)
            if m is None:
                continue
            X = g[ATRIBUTOS].to_numpy(dtype=float)
            Z = (X[:, None, :] - m["med"][None]) / m["esc"][None]
            D = np.nan_to_num(np.where(sentido, np.abs(Z), np.clip(Z, 0, None)))
            if limite is not None and r in limite.index:
                D = D / limite.loc[r, ATRIBUTOS].to_numpy(dtype=float)
            score = np.where(self._admissiveis(m, g), D.max(axis=2), np.inf)
            j = score.argmin(axis=1)
            z.loc[g.index] = Z[np.arange(len(g)), j]
            reg.loc[g.index] = j
        return z, reg

    def z(self, p, limite=None):
        return self._escolha(p, limite)[0]

    def regime(self, p, limite=None):
        """Nome do regime em que cada período melhor se encaixa."""
        _, reg = self._escolha(p, limite)
        return pd.Series([self.rec[r]["nomes"][j] if j >= 0 else None for r, j in zip(p["recurso"], reg)], index=p.index)

    def admissiveis(self, p):
        """Nomes dos regimes admissíveis no contexto de cada período."""
        out = []
        for i, row in p.iterrows():
            m = self.rec.get(row["recurso"])
            a = self._admissiveis(m, p.loc[[i]])[0] if m else []
            out.append([m["nomes"][j] for j in np.flatnonzero(a)] if m else [])
        return out

    def referencia(self, p, limite=None):
        _, reg = self._escolha(p, limite)
        saida = []
        for chave in ("med", "esc", "q05", "q95"):
            v = np.full((len(p), len(ATRIBUTOS)), np.nan)
            for n_, (r, j) in enumerate(zip(p["recurso"], reg)):
                if j >= 0:
                    v[n_] = self.rec[r][chave][j]
            saida.append(pd.DataFrame(v, index=p.index, columns=ATRIBUTOS))
        return saida


METODOS = {
    "Regimes normais (proposto)": lambda: Regimes(contextual=True),
    "Perfil por contexto": lambda: Perfil(("tipo_dia", "hora")),
    "Regimes sem contexto": lambda: Regimes(contextual=False),
    "z-score global": lambda: Perfil(()),
    "Semana anterior": lambda: SemanaAnterior(),
}
PROPOSTO = "Regimes normais (proposto)"


def com_semana_anterior(periodos, jan_ref):
    """Acrescenta ``ant_<atributo>`` (mesmo período 7 dias antes) a períodos alinhados."""
    ref = jan_ref.set_index(["recurso", "janela"])[ATRIBUTOS]
    ant = ref.reindex(pd.MultiIndex.from_arrays([periodos["recurso"], periodos["janela"] - pd.Timedelta("7D")]))
    for a in ATRIBUTOS:
        periodos[f"ant_{a}"] = ant[a].to_numpy()
    return periodos


## 5. Limite por atributo e decisão Normal/Residual
- Para cada método, tamanho de faixa L, mês de teste e histórico (3 ou 1 mês), os escores do histórico são obtidos **fora da amostra** (validação cruzada com 4 blocos contíguos de tempo: ajusta em 3, avalia o restante).
- **Limite do atributo a no recurso r:** percentil 95 das suas variações (mínimo 3), com c = 1. Nos regimes, o regime em que o período se encaixa depende dos limites; por isso a calibração tem uma 2ª passada com os limites da 1ª.
- **Residual** se algum atributo passa do seu limite (nos regimes: em todos os regimes admissíveis). **Indicador responsável** = atributo que mais passou do limite (em múltiplos do limite), no regime admissível mais próximo.


In [ ]:
def dobras(p, periodo, k=4):
    """Blocos contíguos de tempo de igual duração dentro do período (validação cruzada)."""
    frac = (p["janela"] - periodo[0]) / (periodo[1] - periodo[0])
    return np.floor(frac * k).clip(0, k - 1).astype(int).to_numpy()


def calibrar(jan1h, periodos_L, fab, periodo, meta=META_ALARMES):
    """Limites por recurso × atributo para períodos de um tamanho L, com histórico ``periodo``."""
    tr1h = recortar(jan1h, periodo)
    p = recortar(periodos_L, periodo)
    dobra_p, dobra_1h = dobras(p, periodo), dobras(tr1h, periodo)
    rec = p["recurso"].to_numpy()
    modelos = [fab().ajustar(tr1h[dobra_1h != k], p[dobra_p != k]) for k in range(4)]

    def desvios(limite=None):
        d = pd.DataFrame(np.nan, index=p.index, columns=ATRIBUTOS)
        for k, m in enumerate(modelos):
            d.loc[dobra_p == k] = z_direcional(m.z(p[dobra_p == k], limite)).to_numpy()
        if isinstance(modelos[0], SemanaAnterior):
            d[p["ant_log_vol"].isna().to_numpy()] = np.nan  # sem a semana anterior
        return d

    d = desvios()
    base = d.groupby(rec).quantile(Q_LIMITE).clip(lower=LIMITE_MIN)
    if getattr(modelos[0], "escolhe_regime", False):
        # o regime em que o período se encaixa depende dos limites: 2ª passada com os limites da 1ª
        d = desvios(base)
        base = d.groupby(rec).quantile(Q_LIMITE).clip(lower=LIMITE_MIN)
    razao = (d / base.reindex(rec).to_numpy()).max(axis=1, skipna=False).dropna()
    c = 1.0 if meta is None else max(1.0, float(np.quantile(razao, 1 - meta)))
    modelo = fab().ajustar(tr1h, p)
    return dict(modelo=modelo, limite=base * c, c=c, base=base, razao_cal=razao, rec_cal=p.loc[razao.index, "recurso"])


def decidir(p, cal):
    """Estado e indicador de cada período (linhas de ``p``) com a calibração ``cal``."""
    d = z_direcional(cal["modelo"].z(p, cal["limite"]))
    vezes = d / cal["limite"].reindex(p["recurso"]).to_numpy()
    residual = (vezes > 1).any(axis=1)
    indicador = vezes.idxmax(axis=1).map(INDICADOR).where(residual)
    return residual.to_numpy(), indicador, vezes


def abrangencia(linha, indicador):
    """Consumidor específico: o cliente com mais erros concentra ≥ 80% deles e ≥ 20 p.p. acima da sua fatia do volume."""
    fam = {"erros 4xx": "4xx", "erros 5xx": "5xx"}.get(indicador)
    if fam is None:
        return None
    if linha["n_cli"] < 3:
        return "indeterminada"
    top_err, top_vol = linha[f"top_err_{fam}"], linha[f"top_vol_{fam}"]
    return "consumidor específico" if top_err >= 0.8 and top_err - top_vol >= 0.2 else "generalizado"


## 6. Cenários de incidente: injeção de falhas
Cada cenário é independente: uma degradação começa em t₀ = t_reg − atraso e continua até o registro. A falha é aplicada nos registros por minuto × cliente do recurso, numa cópia dos dados.

In [ ]:
def aplicar_motivo(f, t0, motivo, intensidade, rng):
    """Aplica o motivo aos registros com t ≥ t0 da fatia ``f`` (cópia)."""
    f = {k: v.copy() for k, v in f.items()}
    t0m = minutos_desde(pd.Series([t0]))[0]
    m = f["t"] >= t0m
    if f["qtd"][m].sum() == 0:  # sem tráfego no intervalo degradado: a falha não tem como se manifestar
        return f
    fator = MOTIVOS[motivo][2][intensidade]
    if motivo == "latencia":
        f["tmp"][m] *= fator
        return f
    sel = m & (f["status"] // 100 == 2)
    if motivo.endswith("cliente"):
        vol = np.bincount(f["cli"][m], weights=f["qtd"][m])
        alvo = rng.choice(len(vol), p=vol / vol.sum())
        sel &= f["cli"] == alvo
    mov = rng.binomial(f["qtd"][sel], fator)
    f["qtd"][sel] -= mov
    novo_status = 400 if motivo.startswith("4xx") else 503
    add = {k: v[sel].copy() for k, v in f.items()}
    add["qtd"], add["status"] = mov, np.full(sel.sum(), novo_status)
    if novo_status == 400:
        add["tmp"] = add["tmp"] * 0.5  # rejeição de negócio costuma ser mais rápida
    return {k: np.concatenate([f[k], add[k]]) for k in f}


def elegiveis(brutos):
    """Recurso × mês de teste com ao menos MIN_HIST_DIAS dias de dados no histórico de 1 mês (e portanto no de 3)."""
    linhas = []
    for mes, (ini, fim) in MESES_TESTE.items():
        h_ini = ini - pd.DateOffset(months=1)
        for r, a in brutos.por_recurso.items():
            dias = len(np.unique((a["t"][(a["t"] >= minutos_desde(pd.Series([h_ini]))[0]) & (a["t"] < minutos_desde(pd.Series([ini]))[0])]) // 1440))
            t_ini = max(ini, brutos.inicio[r]) + pd.Timedelta(minutes=H_MAX)
            t_fim = min(fim, brutos.fim[r])
            linhas.append(dict(mes=mes, recurso=r, dias_hist_1m=dias, t_ini=t_ini, t_fim=t_fim,
                               elegivel=dias >= MIN_HIST_DIAS and t_fim > t_ini))
    return pd.DataFrame(linhas)


def gerar_cenarios(eleg, semente=SEMENTE, repeticoes=REPETICOES):
    """Por mês × recurso elegível: incidentes (motivo × intensidade × atraso × repetição) e o mesmo número de controles."""
    rng = np.random.default_rng(semente)
    linhas = []
    for e in eleg[eleg["elegivel"]].itertuples():
        n_min = int((e.t_fim - e.t_ini) / pd.Timedelta("1min"))
        sorteia = lambda: e.t_ini + pd.Timedelta(minutes=int(rng.integers(n_min)))
        n_inc = 0
        for motivo, (_, _, niveis) in MOTIVOS.items():
            for intens in niveis:
                for atraso in ATRASOS:
                    for _ in range(repeticoes):
                        linhas.append(dict(mes=e.mes, tipo="incidente", recurso=e.recurso, motivo=motivo,
                                           intensidade=intens, atraso=atraso, t_registro=sorteia()))
                        n_inc += 1
        for _ in range(n_inc):
            linhas.append(dict(mes=e.mes, tipo="controle", recurso=e.recurso, motivo=None, intensidade=None,
                               atraso=None, t_registro=sorteia()))
    c = pd.DataFrame(linhas)
    c.insert(0, "cenario", range(len(c)))
    return c

## 7. Períodos de cada cenário e expansão retroativa
Para cada cenário: as faixas de decisão [t − L, t) com L = 15, 30 e 60 min e as faixas da expansão retroativa, além dos mesmos períodos 7 dias antes (para o baseline semana anterior).

In [ ]:
def periodos_do_cenario(brutos, cen, semente=SEMENTE):
    rng = np.random.default_rng(semente * 1_000_003 + int(cen.cenario))
    t = pd.Timestamp(cen.t_registro)
    f = brutos.fatia(cen.recurso, t - pd.Timedelta(minutes=H_MAX), t)
    sem_trafego = False
    if cen.tipo == "incidente":
        t0 = t - pd.Timedelta(minutes=int(cen.atraso))
        sem_trafego = bool(f["qtd"][f["t"] >= minutos_desde(pd.Series([t0]))[0]].sum() == 0)
        f = aplicar_motivo(f, t0, cen.motivo, cen.intensidade, rng)
    f_ant = brutos.fatia(cen.recurso, t - pd.Timedelta(days=7, minutes=H_MAX), t - pd.Timedelta(days=7))
    pedidos = [("L", L, 0, L) for L in FAIXAS_L] + [("banda", i, lo, hi) for i, (lo, hi) in enumerate(BANDAS)]
    linhas = []
    for tipo_p, id_p, lo, hi in pedidos:
        ini, fim = t - pd.Timedelta(minutes=hi), t - pd.Timedelta(minutes=lo)
        o = agregar_periodo(f, ini, fim)
        a = agregar_periodo(f_ant, ini - pd.Timedelta(days=7), fim - pd.Timedelta(days=7))
        o.update({f"ant_{k}": a[k] for k in ATRIBUTOS})
        o.update(cenario=cen.cenario, recurso=cen.recurso, periodo=tipo_p, id_periodo=id_p, lo=lo, hi=hi,
                 sem_trafego=sem_trafego)
        linhas.append(o)
    return linhas


def estimar_inicio(res_bandas):
    """``res_bandas``: residual de cada banda (ordem de BANDAS). Retorna (estado, lo, hi) da zona de transição
    em minutos antes do registro: começou entre ``hi`` e ``lo`` minutos antes."""
    if not res_bandas[0]:
        return "Normal", None, None
    for j in range(1, len(BANDAS)):
        if not res_bandas[j]:
            return "Residual", BANDAS[j - 1][0], BANDAS[j][1]
    return "Residual", BANDAS[-1][0], None  # não voltou ao normal em 4 h

## 8. Estatística: IC de Wilson e McNemar

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1 + z ** 2 / n
    centro = (p + z ** 2 / (2 * n)) / den
    meia = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return (centro - meia, centro + meia)


def fmt_ic(k, n):
    lo, hi = wilson(k, n)
    return f"{k / n:.1%} [{lo:.1%}; {hi:.1%}]" if n else "—"


def mcnemar(acerto_a, acerto_b):
    """Teste exato de McNemar sobre acertos pareados (mesmos casos)."""
    a, b = np.asarray(acerto_a, bool), np.asarray(acerto_b, bool)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    p = stats.binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return n01, n10, p

## 9. Estilo dos gráficos

In [ ]:
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
AZUL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
TXT1, TXT2, TXT3, GRADE, CINZA = "#0b0b0b", "#52514e", "#8a8983", "#e4e3df", "#b8b7b1"
CMAP_SEQ = LinearSegmentedColormap.from_list("azul", ["#f7f9fc"] + AZUL)
COR_DIA = {"util": CORES[0], "nao_util": CORES[1]}
mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8.5, "axes.titleweight": "bold",
    "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.edgecolor": CINZA, "axes.labelcolor": TXT2, "xtick.color": TXT2, "ytick.color": TXT2,
    "text.color": TXT1, "axes.titlecolor": TXT1, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.5, "figure.dpi": 110, "savefig.bbox": "tight", "savefig.dpi": 220,
    "pdf.fonttype": 42, "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
})


def salvar(fig, nome):
    fig.savefig(FIGURAS / f"{nome}.pdf")
    fig.savefig(FIGURAS / f"{nome}.png")


def salvar_json(obj, nome):
    (RESULTADOS / nome).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str))


def heatmap(ax, v, fmt="{:.0%}", vmax=1, limiar_branco=0.6):
    ax.imshow(v.to_numpy(dtype=float), cmap=CMAP_SEQ, vmin=0, vmax=vmax, aspect="auto")
    for i in range(v.shape[0]):
        for j in range(v.shape[1]):
            x = v.iat[i, j]
            if pd.notna(x):
                ax.text(j, i, fmt.format(x), ha="center", va="center", fontsize=6.8,
                        color="white" if x / vmax > limiar_branco else TXT1)
    ax.set_xticks(range(v.shape[1]))
    ax.set_xticklabels(v.columns)
    ax.set_yticks(range(v.shape[0]))
    ax.set_yticklabels(v.index)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

print("00_funcoes (v7) carregado ·", VERSAO)